In [3]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [5]:
client.search_experiments()

[<Experiment: artifact_location='/home/vagrant/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1', creation_time=1744933790508, experiment_id='1', last_update_time=1744933790508, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1744933717780, experiment_id='0', last_update_time=1744933717780, lifecycle_stage='active', name='Default', tags={}>]

In [6]:
client.create_experiment(name="my-cool-experiment")

'2'

In [15]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids=["1"],
    filter_string="metrics.rmse < 6.7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.accuracy ASC"]
)  

In [16]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")


run id: 5e137f3175ec471da918381d5e106d69, rmse: 6.3213
run id: 4a669f9609f14d8a83f72a327ce784a1, rmse: 6.5564
run id: 3017270461a840f2a1feb2e0c2d2f240, rmse: 6.4494
run id: da7607bcfcec4aa582327fff00765a7f, rmse: 6.3826
run id: 710870cfbeac40f69951cccb22f3c778, rmse: 6.3213


In [17]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [18]:
run_id = "3017270461a840f2a1feb2e0c2d2f240"
model_uri = f"runs:/{run_id}/model"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1745093820948, current_stage='None', description=None, last_updated_timestamp=1745093820948, name='nyc-taxi-regressor', run_id='3017270461a840f2a1feb2e0c2d2f240', run_link=None, source='/home/vagrant/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1/3017270461a840f2a1feb2e0c2d2f240/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [19]:
model_name = "nyc-taxi-regressor"
latest_version = client.get_latest_versions(name=model_name)

for version in latest_version:
    print(f"Version: {version.version}, Stage: {version.current_stage}")
    

Version: 3, Stage: None


/tmp/ipykernel_4039/1801236025.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions(name=model_name)


In [25]:
model_version = 3
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)


/tmp/ipykernel_4039/4126000956.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1745093820948, current_stage='Staging', description=None, last_updated_timestamp=1745095104248, name='nyc-taxi-regressor', run_id='3017270461a840f2a1feb2e0c2d2f240', run_link=None, source='/home/vagrant/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1/3017270461a840f2a1feb2e0c2d2f240/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [27]:
from datetime import datetime

date =datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} is now in {new_stage} stage of as {date}."
)

<ModelVersion: aliases=[], creation_timestamp=1745093820948, current_stage='Staging', description='The model version 3 is now in Staging stage of as 2025-04-19.', last_updated_timestamp=1745095144467, name='nyc-taxi-regressor', run_id='3017270461a840f2a1feb2e0c2d2f240', run_link=None, source='/home/vagrant/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1/3017270461a840f2a1feb2e0c2d2f240/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=3>